# Stacks

```{contents}
:local:
:depth: 2
```


```{index} stack; LIFO
```

## Stack Behavior



A **stack** is an abstract data type where the most recently added item is the next item removed. This access rule is called **last in, first out**, or **LIFO**.

The core operations are small:

- `Push(value)` adds a value to the top.
- `Pop()` removes and returns the top value.
- `Peek()` returns the top value without removing it.
- `Count` reports how many values are currently stored.

The point of a stack is not that it stores data in a new magical way. The point is that it restricts access so the program can only work with the top item.


In [ ]:
using System;
using System.Collections.Generic;

var undoHistory = new Stack<string>();
undoHistory.Push("Open report");
undoHistory.Push("Add chart");
undoHistory.Push("Save report");

Console.WriteLine($"Undo now: {undoHistory.Pop()}");
Console.WriteLine($"Next undo: {undoHistory.Peek()}");
Console.WriteLine($"Actions left: {undoHistory.Count}");


The last action pushed was `Save report`, so it is the first action popped. The stack keeps the order of pending undo actions aligned with how users expect undo to work.


```{index} stack; underflow
```

## Empty Stacks and Safe Access



Calling `Pop` or `Peek` on an empty stack is an **underflow** error. In C#, `Stack<T>` throws an exception if the stack is empty. Production code should usually check `Count` before removing an item, or use `TryPop` and `TryPeek` when available.


In [ ]:
using System;
using System.Collections.Generic;

var pages = new Stack<string>();
pages.Push("home");
pages.Push("products");

while (pages.Count > 0)
{
    string previousPage = pages.Pop();
    Console.WriteLine($"Back to {previousPage}");
}

Console.WriteLine("No more page history.");


The loop checks `Count` before each `Pop`, so it never tries to remove a value after the stack has become empty.


```{index} stack; delimiter matching
```

## Matching Delimiters



Stacks are useful for nested structures because the most recent opening item must be closed first. Parentheses, brackets, braces, HTML tags, and method calls all have this nested shape.


In [ ]:
using System;
using System.Collections.Generic;
foreach (string text in new[] { "{ items[0] + (total - 1) }", "([)]", ")(", "((", "" })
    Console.WriteLine($"{text}: {IsBalanced(text)}");

static bool IsBalanced(string text)
{
    var openings = new Stack<char>();
    foreach (char ch in text)
    {
        if (ch == '(' || ch == '[' || ch == '{') openings.Push(ch);
        else if (ch == ')' || ch == ']' || ch == '}')
        {
            if (!openings.TryPop(out char open)) return false;
            bool matches = (open == '(' && ch == ')')
                        || (open == '[' && ch == ']')
                        || (open == '{' && ch == '}');
            if (!matches) return false;
        }
    }
    return openings.Count == 0;
}

The stack stores exactly the unmatched opening delimiters, with the most recent one on top. A mismatched type or a closing delimiter with no opening returns false immediately. Success also requires an empty stack at the end. This checks delimiter characters only; it does not parse C# strings or comments.

**Trace:** For `([)]`, show the stack after each character until rejection. Why would merely counting openings and closings accept some invalid expressions? Each character is processed once, giving O(n) time and O(d) auxiliary space for maximum nesting depth d.


```{index} stack; implementation
```

## Stack Implementation Choices



A stack can be implemented with an array, a dynamic array, or linked nodes. The public behavior stays the same, but the representation affects growth, memory use, and constant factors.

An array-backed stack usually stores the top item at `items[count - 1]`. Pushing appends at the end; popping reads that end slot and decrements `count`.


In [ ]:
using System;

var stack = new SmallStack();
stack.Push(10);
stack.Push(20);
Console.WriteLine(stack.Pop());
Console.WriteLine(stack.Pop());

class SmallStack
{
    private int[] items = new int[4];
    private int count;

    public int Count => count;

    public void Push(int value)
    {
        if (count == items.Length)
        {
            throw new InvalidOperationException("Stack is full.");
        }

        items[count] = value;
        count++;
    }

    public int Pop()
    {
        if (count == 0)
        {
            throw new InvalidOperationException("Stack is empty.");
        }

        count--;
        return items[count];
    }
}


This tiny implementation is bounded: it cannot grow past four values. A production dynamic-array stack would resize when full, much like the dynamic arrays in the previous chapter.


```{index} trace the stack invariant
```

## Trace the Stack Invariant



A report editor saves versions in order. With count three, the live array slots are `0..2`, and the top is index two. `Peek` must preserve count; `Pop` removes only that top value.

| Operation | Bottom to top | Count | Returned value |
| --- | --- | ---: | --- |
| Push Draft | Draft | 1 | None |
| Push Reviewed | Draft, Reviewed | 2 | None |
| Peek | Draft, Reviewed | 2 | Reviewed |
| Pop | Draft | 1 | Reviewed |
| Pop | Empty | 0 | Draft |

An invariant is a rule preserved after every completed operation. Here `0 <= Count <= Capacity`, the live values occupy consecutive slots, and `items[Count - 1]` is the top when count is positive. An empty pop must reject the request before decrementing count. Zero or null can be valid data; count determines emptiness.

**Predict:** Push A, push B, pop, push C, then peek. Give the returned values and final count. Which slots are live?


```{index} a complete generic array stack
```

## A Complete Generic Array Stack



Chapter 15's dynamic array becomes a stack by exposing only the final live slot. This implementation doubles capacity when full and clears popped slots so unused storage does not retain object references. `default!` suppresses a nullable warning for unused storage; it is not a runtime check.

`TryPop` returns false on empty input and places a default value in its output parameter. Always check the Boolean: a default value could also have been successfully popped. Expect Approved, Reviewed, Draft, followed by `empty=True`.

**Extend:** Add `TryPeek` with the same success rule. Test empty, singleton, duplicate, and zero-valued stacks without changing count.

For local VS Code practice, download {download}`the complete implementations and verification program <../../materials/16/ServiceDeskStructures.cs>` and use it as `Program.cs` in a console project. The program compares stack/queue behavior with standard library collections and checks boundary cases.


In [ ]:
using System;
var history = new ArrayStack<string>();
history.Push("Draft"); history.Push("Reviewed"); history.Push("Approved");
Console.WriteLine($"count={history.Count}, capacity={history.Capacity}, top={history.Peek()}");
while (history.TryPop(out string state)) Console.WriteLine(state);
Console.WriteLine($"empty={history.Count == 0}");

public class ArrayStack<T>
{
    private T[] items = new T[2];
    public int Count { get; private set; }
    public int Capacity => items.Length;
    public void Push(T value)
    {
        if (Count == Capacity)
        {
            var larger = new T[Capacity * 2];
            Array.Copy(items, larger, Count);
            items = larger;
        }
        items[Count++] = value;
    }
    public T Peek()
    {
        if (Count == 0) throw new InvalidOperationException("Stack is empty.");
        return items[Count - 1];
    }
    public T Pop()
    {
        T value = Peek();
        items[--Count] = default!;
        return value;
    }
    public bool TryPop(out T value)
    {
        if (Count == 0) { value = default!; return false; }
        value = Pop();
        return true;
    }
}

```{index} correctness and cost of stack operations
```

## Correctness and Cost of Stack Operations



`Push` preserves the invariant by growing without changing count, writing to the first unused slot, and increasing count once. `Pop` first obtains the current top, then reduces count and clears precisely the slot that left the live range. It never shifts other values.

Pop and peek take O(1) time. Push takes O(1) on most calls, O(n) when growth copies n live values, and O(1) amortized across repeated pushes using the chapter 15 doubling argument. A linked stack can push and pop at the head in O(1) link operations, but allocates a node per push. Array locality and spare capacity are different memory tradeoffs.

**Check:** For five pushes starting at capacity two, count the existing entries copied. Explain why removing every item from this array stack needs only n pops, while repeatedly removing index zero from an array sequence moves `n(n - 1)/2` entries.


```{index} stack; exercise
```

## Exercise



In [ ]:
// Exercise: Reverse the order
// Fill the stack with the three task names, then pop and print them so
// the output is Deploy, Test, Build.

using System;
using System.Collections.Generic;

var tasks = new Stack<string>();

// Your code starts here.



// Your code ends here.


In [1]:
// Solution
using System;
using System.Collections.Generic;

var tasks = new Stack<string>();
tasks.Push("Build");
tasks.Push("Test");
tasks.Push("Deploy");

while (tasks.Count > 0)
{
    Console.WriteLine(tasks.Pop());
}


Deploy
Test
Build


```{index} summary
```

## Summary



A stack is the right structure when the next item to process is the most recent item saved. C# provides `Stack<T>` for ordinary use, while small implementations help reveal why push and pop are naturally end-of-sequence operations.


```{rubric} Footnotes
```

[^1]: C# `Stack<T>` is part of `System.Collections.Generic`.
